This script preprocesses CORINE Land Cover data for Bavaria by clipping all land cover classes to the Bavarian boundary, merging them into a single dataset for each reference year, and exporting one GeoPackage per year. The 2021 Land Cover Model (LBM) is processed separately to account for its different data structure before being harmonized with the CORINE datasets.

In [ ]:
"""
CORINE Land Cover Preprocessing for Bavaria

Processes CORINE Land Cover data:
1. Clips all class files (class1xx, class2xx, class3xx, etc.) to Bavaria
2. Merges all classes per year into a single GPKG file
3. Creates one GPKG per year with all land cover classes

Input structure:
  clc5_YYYY.utm32s.shape/clc5/
    ├── clc5_class1xx.shp (Artificial surfaces)
    ├── clc5_class2xx.shp (Agricultural areas)
    ├── clc5_class3xx.shp (Forest and semi-natural areas)
    ├── clc5_class4xx.shp (Wetlands)
    └── clc5_class5xx.shp (Water bodies)

Output:
  CORINE_Bavaria_YYYY.gpkg (one file per year)

Author: Agnes Zwick
Date: December 2025
"""

import geopandas as gpd
from pathlib import Path
import pandas as pd
from datetime import datetime

from pathlib import Path
import geopandas as gpd


def find_corine_files(corine_base_dir, year):
    """
    Finds all CORINE class shapefiles for a given year and detects the CLC column automatically.

    Handles two different structures:
    - 2012, 2015, 2018:
      clc5_YYYY.utm32s.shape/clc5/clc5_classXxx.shp

    - 2021 (Landbedeckungsmodell):
      lbm-de2021.utm32s.shape/lbm-de2021/land/by/lbm-de2021_*.shp

    Parameters
    ----------
    corine_base_dir : str or Path
        Base directory containing CORINE data
    year : int
        Year (e.g. 2012, 2015, 2018, 2021)

    Returns
    -------
    dict
        {
            'files': list[Path],
            'clc_column': str
        }
    """

    base_path = Path(corine_base_dir)

    # -------------------------
    # Find files
    # -------------------------
    if year == 2021:
        lbm_dir = (
            base_path
            / "lbm-de2021.utm32s.shape"
            / "lbm-de2021"
            / "land"
            / "by"
        )

        if not lbm_dir.exists():
            raise FileNotFoundError(f"Directory not found: {lbm_dir}")

        files = sorted(lbm_dir.glob("lbm-de2021_*.shp"))

    else:
        year_dir = base_path / f"clc5_{year}.utm32s.shape" / "clc5"

        if not year_dir.exists():
            raise FileNotFoundError(f"Directory not found: {year_dir}")

        files = sorted(year_dir.glob("clc5_class*.shp"))

    if not files:
        raise FileNotFoundError(f"No CORINE shapefiles found for year {year}")

    # -------------------------
    # Detect CLC column
    # -------------------------
    gdf_sample = gpd.read_file(files[0])

    clc_cols = [c for c in gdf_sample.columns if c.upper().startswith("CLC")]

    if len(clc_cols) == 0:
        raise ValueError("No CLC column found in CORINE data")
    if len(clc_cols) > 1:
        raise ValueError(f"Multiple CLC columns found: {clc_cols}")

    clc_column = clc_cols[0]

    return {
        "files": files,
        "clc_column": clc_column
    }


def load_bavaria_boundary(boundary_file, boundary_layer="vg250_lan"):
    """
    Loads Bavaria boundary from VG250
    
    Parameters:
    -----------
    boundary_file : str/Path
        Path to VG250 GPKG file
    boundary_layer : str
        Layer name (default: vg250_lan for federal states)
    
    Returns:
    --------
    GeoDataFrame: Bavaria boundary
    """
    print("Loading Bavaria boundary...")
    
    gdf = gpd.read_file(boundary_file, layer=boundary_layer)
    
    # Filter for Bavaria (GF=4 means Bayern, or use AGS starting with 09)
    bavaria = gdf[gdf['GEN'] == "Bayern"].copy()
    
    if len(bavaria) == 0:
        raise ValueError("Bavaria not found in boundary file!")
    
    print(f"  ✓ Bavaria boundary loaded")
    print(f"    CRS: {bavaria.crs}")
    
    return bavaria


def process_corine_year(
    corine_base_dir,
    year,
    bavaria_boundary,
    output_file,
    verbose=True
):
    """
    Processes all CORINE classes for one year
    
    Parameters:
    -----------
    corine_base_dir : str/Path
        Base directory with CORINE data
    year : int
        Year to process
    bavaria_boundary : GeoDataFrame
        Bavaria boundary for clipping
    output_file : str/Path
        Output GPKG file path
    verbose : bool
        Detailed output
    
    Returns:
    --------
    GeoDataFrame: Merged and clipped CORINE data for Bavaria
    """
    
    print(f"\n{'='*80}")
    print(f"Processing CORINE {year}")
    print(f"{'='*80}")
    
    # Find all class files
    files_info = find_corine_files(corine_base_dir, year)
    class_files = files_info['files']
    clc_column = files_info['clc_column']
    
    if len(class_files) == 0:
        print(f"  No class files found for {year}")
        return None
    
    print(f"Using column: {clc_column}")
    print(f"Found {len(class_files)} class files:")
    for f in class_files:
        print(f"  - {Path(f).name}")
    
    # Load and process each class
    all_classes = []
    
    for idx, class_file in enumerate(class_files, 1):
        class_name = Path(class_file).stem
        
        if verbose:
            print(f"\n[{idx}/{len(class_files)}] Processing {class_name}...")
        
        try:
            # Load shapefile
            gdf = gpd.read_file(class_file)
            
            if verbose:
                print(f"  Loaded: {len(gdf):,} features")
                print(f"  CRS: {gdf.crs}")
            
            # Check for the year-specific CLC column
            if clc_column not in gdf.columns:
                print(f"     Warning: '{clc_column}' column not found!")
                print(f"     Available columns: {', '.join(gdf.columns)}")
                # Try to find any CLC column
                clc_cols = [col for col in gdf.columns if col.startswith('CLC')]
                if clc_cols:
                    found_col = clc_cols[0]
                    print(f"     Using column: {found_col}")
                    gdf['CLC_CODE'] = gdf[found_col]
                else:
                    print(f"     Skipping {class_name} - no CLC column found")
                    continue
            else:
                # Use the correct column
                gdf['CLC_CODE'] = gdf[clc_column]
            
            # Ensure same CRS as Bavaria boundary
            if gdf.crs != bavaria_boundary.crs:
                if verbose:
                    print(f"  Reprojecting to {bavaria_boundary.crs}...")
                gdf = gdf.to_crs(bavaria_boundary.crs)
            
            # Clip to Bavaria
            if verbose:
                print(f"  Clipping to Bavaria...")
            
            gdf_clipped = gpd.clip(gdf, bavaria_boundary)
            
            if verbose:
                print(f"  After clipping: {len(gdf_clipped):,} features")
            
            if len(gdf_clipped) > 0:
                all_classes.append(gdf_clipped)
            
        except Exception as e:
            print(f"    Error processing {class_name}: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    if len(all_classes) == 0:
        print(f"\n  No valid data for {year}")
        return None
    
    # Merge all classes
    print(f"\n{'='*80}")
    print(f"Merging {len(all_classes)} classes...")
    
    merged = pd.concat(all_classes, ignore_index=True)
    merged = gpd.GeoDataFrame(merged, crs=all_classes[0].crs)
    
    print(f"  Total features: {len(merged):,}")
    
    # Show class distribution
    if 'CLC_CODE' in merged.columns:
        print(f"\nLand cover classes found:")
        class_counts = merged['CLC_CODE'].value_counts().sort_index()
        for clc_code, count in class_counts.head(20).items():
            print(f"  {clc_code}: {count:,} features")
        if len(class_counts) > 20:
            print(f"  ... and {len(class_counts)-20} more classes")
    
    # Save
    print(f"\nSaving to: {output_file}")
    output_path = Path(output_file)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    merged.to_file(output_file, driver='GPKG')
    print(f"  ✓ Saved: {output_file}")
    
    # Calculate total area
    total_area_m2 = merged.geometry.area.sum()
    total_area_km2 = total_area_m2 / 1_000_000
    print(f"  Total area: {total_area_km2:,.1f} km²")
    
    return merged


def process_lbm_2021(
    lbm_base_dir,
    output_file,
    verbose=True
):
    """
    Processes LBM 2021 data (special format with Nord/Süd files for Bavaria)
    
    LBM 2021 has a different structure:
    - lbm-de2021.utm32s.shape/lbm-de2021/land/by/lbm-de2021_nord.shp
    - lbm-de2021.utm32s.shape/lbm-de2021/land/by/lbm-de2021_sued.shp
    - Uses CLC21 column
    
    NEW: Dissolves by CLC_CODE to merge adjacent polygons with same class
    
    Parameters:
    -----------
    lbm_base_dir : str/Path
        Base directory with LBM 2021 data
    output_file : str/Path
        Output GPKG file path
    verbose : bool
        Detailed output
    
    Returns:
    --------
    GeoDataFrame: Merged and dissolved LBM 2021 data for Bavaria
    """
    
    print(f"\n{'='*80}")
    print(f"Processing LBM 2021 (Special Format)")
    print(f"{'='*80}")
    print(f"Using column: CLC21")
    
    lbm_path = Path(lbm_base_dir)
    
    # Path to Bavaria files
    by_path = lbm_path / "lbm-de2021.utm32s.shape" / "lbm-de2021" / "land" / "by"
    
    if not by_path.exists():
        print(f"  Bavaria directory not found: {by_path}")
        return None
    
    # Find Nord and Süd files
    nord_file = by_path / "lbm-de2021_nord.shp"
    sued_file = by_path / "lbm-de2021_sued.shp"
    
    files_to_load = []
    if nord_file.exists():
        files_to_load.append(('Nord', nord_file))
    else:
        print(f"   Nord file not found: {nord_file}")
    
    if sued_file.exists():
        files_to_load.append(('Süd', sued_file))
    else:
        print(f"   Süd file not found: {sued_file}")
    
    if len(files_to_load) == 0:
        print(f"  No files found!")
        return None
    
    print(f"Found {len(files_to_load)} files:")
    for name, filepath in files_to_load:
        print(f"  - {name}: {filepath.name}")
    
    # Load and merge
    parts = []
    
    for name, filepath in files_to_load:
        if verbose:
            print(f"\nLoading {name}...")
        
        try:
            gdf = gpd.read_file(filepath)
            
            if verbose:
                print(f"  Loaded: {len(gdf):,} features")
                print(f"  CRS: {gdf.crs}")
                print(f"  Columns: {', '.join(gdf.columns[:10])}...")
            
            # Check for CLC21 column
            if 'CLC21' not in gdf.columns:
                print(f"     Warning: CLC21 column not found!")
                print(f"     Available columns: {', '.join(gdf.columns)}")
                # Try to find alternative
                clc_cols = [col for col in gdf.columns if col.startswith('CLC')]
                if clc_cols:
                    found_col = clc_cols[0]
                    print(f"     Using column: {found_col}")
                    gdf['CLC_CODE'] = gdf[found_col]
                else:
                    print(f"     Skipping {name} - no CLC column found")
                    continue
            else:
                # Use CLC21 and rename to CLC_CODE for consistency
                gdf['CLC_CODE'] = gdf['CLC21']
            
            parts.append(gdf)
            
        except Exception as e:
            print(f"    Error loading {name}: {e}")
            continue
    
    if len(parts) == 0:
        print(f"\n  No valid data for 2021")
        return None
    
    # Merge Nord and Süd
    print(f"\n{'='*80}")
    print(f"Merging {len(parts)} parts (Nord/Süd)...")
    
    merged = pd.concat(parts, ignore_index=True)
    merged = gpd.GeoDataFrame(merged, crs=parts[0].crs)
    
    print(f"  Total features before dissolve: {len(merged):,}")
    
    # DISSOLVE by CLC_CODE - combines all polygons with same land cover class
    print(f"\nDissolving by CLC_CODE (this may take a few minutes)...")
    
    # Keep only necessary columns for dissolve (speeds up processing)
    cols_to_keep = ['CLC_CODE', 'geometry']
    merged_simple = merged[cols_to_keep].copy()
    
    # Dissolve
    merged = merged_simple.dissolve(by='CLC_CODE', as_index=False)
    
    print(f"  Total features after dissolve: {len(merged):,}")
    
    # Show class distribution
    if 'CLC_CODE' in merged.columns:
        print(f"\nLand cover classes found:")
        class_counts = merged['CLC_CODE'].value_counts().sort_index()
        for clc_code, count in class_counts.head(20).items():
            print(f"  {clc_code}: {count:,} polygons")
        if len(class_counts) > 20:
            print(f"  ... and {len(class_counts)-20} more classes")
    
    # Save
    print(f"\nSaving to: {output_file}")
    output_path = Path(output_file)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    merged.to_file(output_file, driver='GPKG')
    print(f"  ✓ Saved: {output_file}")
    
    # Calculate total area
    total_area_m2 = merged.geometry.area.sum()
    total_area_km2 = total_area_m2 / 1_000_000
    print(f"  Total area: {total_area_km2:,.1f} km²")
    
    return merged


def process_all_corine_years(
    corine_base_dir,
    lbm_base_dir,
    boundary_file,
    output_dir,
    years=[2012, 2015, 2018, 2021],
    boundary_layer="vg250_lan",
    skip_existing=True
):
    """
    Processes all CORINE years (including LBM 2021 with special handling)
    
    Parameters:
    -----------
    corine_base_dir : str/Path
        Base directory with CORINE data (2012, 2015, 2018)
    lbm_base_dir : str/Path
        Base directory with LBM 2021 data
    boundary_file : str/Path
        Path to VG250 boundary file
    output_dir : str/Path
        Output directory
    years : list
        List of years to process
    boundary_layer : str
        Boundary layer name
    skip_existing : bool
        If True, skip years where output file already exists
    
    Returns:
    --------
    dict: {year: output_file}
    """
    
    print(f"\n{'#'*80}")
    print("CORINE/LBM LAND COVER PREPROCESSING FOR BAVARIA")
    print(f"{'#'*80}")
    print(f"CORINE Input: {corine_base_dir}")
    print(f"LBM Input:    {lbm_base_dir}")
    print(f"Output:       {output_dir}")
    print(f"Years:        {years}")
    print(f"Skip existing: {skip_existing}")
    print(f"{'#'*80}\n")
    
    # Check which years need processing
    years_to_process = []
    
    for year in years:
        output_file = Path(output_dir) / f"CORINE_Bavaria_{year}.gpkg"
        if skip_existing and output_file.exists():
            print(f"{year}: Output file already exists - skipping")
        else:
            years_to_process.append(year)
    
    if len(years_to_process) == 0:
        print("\n✓ All years already processed!")
        # Return existing files
        results = {}
        for year in years:
            output_file = Path(output_dir) / f"CORINE_Bavaria_{year}.gpkg"
            if output_file.exists():
                results[year] = str(output_file)
        return results
    
    print(f"\nYears to process: {years_to_process}")

    # Load Bavaria boundary only if needed
    bavaria = None
    if any(y in [2012, 2015, 2018] for y in years_to_process):
        bavaria = load_bavaria_boundary(boundary_file, boundary_layer)
    
    # Process each year
    results = {}
    successful = 0
    failed = 0
    skipped = len(years) - len(years_to_process)  # Already skipped earlier
    
    for year in years_to_process:
        output_file = Path(output_dir) / f"CORINE_Bavaria_{year}.gpkg"
        
        try:
            if year == 2021:
                # Special processing for LBM 2021
                result = process_lbm_2021(
                    lbm_base_dir,
                    output_file,
                    verbose=True
                )
            else:
                # Standard CORINE processing (2012, 2015, 2018)
                result = process_corine_year(
                    corine_base_dir,
                    year,
                    bavaria,
                    output_file,
                    verbose=True
                )
            
            if result is not None:
                results[year] = str(output_file)
                successful += 1
                print(f"✓ {year}: Success")
            else:
                failed += 1
                print(f"   {year}: No data")
                
        except Exception as e:
            failed += 1
            print(f"  {year}: Error - {e}")
            import traceback
            traceback.print_exc()
            continue
    
    # Add already existing files to results
    for year in years:
        output_file = Path(output_dir) / f"CORINE_Bavaria_{year}.gpkg"
        if year not in results and output_file.exists():
            results[year] = str(output_file)
    
    print(f"\nOutput files:")
    for year in sorted(results.keys()):
        filepath = results[year]
        print(f"  {year}: {Path(filepath).name}")
    
    return results


def check_corine_structure(corine_base_dir, lbm_base_dir=None):
    """
    Checks the structure of CORINE and LBM directories
    
    Parameters:
    -----------
    corine_base_dir : str/Path
        Base directory with CORINE data
    lbm_base_dir : str/Path, optional
        Base directory with LBM 2021 data
    """
    
    print(f"\n{'='*80}")
    print("CHECKING CORINE/LBM DIRECTORY STRUCTURE")
    print(f"{'='*80}")
    
    # Check CORINE
    print(f"\nCORINE Base directory: {corine_base_dir}\n")
    
    base_path = Path(corine_base_dir)
    
    if not base_path.exists():
        print(f"  CORINE directory does not exist!")
    else:
        # Find all year directories
        year_dirs = sorted(base_path.glob("clc5_*.utm32s.shape"))
        
        if len(year_dirs) == 0:
            print(f"  No CORINE year directories found!")
        else:
            print(f"Found {len(year_dirs)} CORINE year directories:\n")
            
            for year_dir in year_dirs:
                year_str = year_dir.name.split('_')[1].split('.')[0]
                
                print(f"{year_str}:")
                print(f"  Directory: {year_dir.name}")
                
                clc5_dir = year_dir / "clc5"
                
                if not clc5_dir.exists():
                    print(f"     clc5 subdirectory not found!")
                    continue
                
                class_files = sorted(clc5_dir.glob("clc5_class*.shp"))
                
                if len(class_files) == 0:
                    print(f"     No class shapefiles found!")
                else:
                    print(f"  ✓ {len(class_files)} class files:")
                    for cf in class_files:
                        print(f"    - {cf.name}")
                
                print()
    
    # Check LBM 2021
    if lbm_base_dir:
        print(f"\n{'='*80}")
        print(f"LBM 2021 directory: {lbm_base_dir}\n")
        
        lbm_path = Path(lbm_base_dir)
        
        if not lbm_path.exists():
            print(f"  LBM directory does not exist!")
        else:
            by_path = lbm_path / "lbm-de2021.utm32s.shape" / "lbm-de2021" / "land" / "by"
            
            print(f"2021 (LBM):")
            print(f"  Expected path: {by_path}")
            
            if not by_path.exists():
                print(f"    Bavaria directory not found!")
            else:
                print(f"  ✓ Bavaria directory found")
                
                nord_file = by_path / "lbm-de2021_nord.shp"
                sued_file = by_path / "lbm-de2021_sued.shp"
                
                files_found = []
                if nord_file.exists():
                    files_found.append("Nord")
                if sued_file.exists():
                    files_found.append("Süd")
                
                if len(files_found) == 0:
                    print(f"    No Nord/Süd files found!")
                else:
                    print(f"  ✓ Files found: {', '.join(files_found)}")
                    for name in files_found:
                        file_path = by_path / f"lbm-de2021_{name.lower()}.shp"
                        print(f"    - lbm-de2021_{name.lower()}.shp")
            
            print()


def main():
    """Main program"""
    
    # ========================================================================
    # CONFIGURATION
    # ========================================================================
    
    # Korrigierter Aufruf in deinem Notebook:

    # Konfiguration
    CORINE_BASE_DIR = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Input\CORINE_Land_Cover_DE"
    LBM_BASE_DIR = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Input\Landbedeckungsmodell_DE"
    BOUNDARY_FILE = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Input\Verwaltungsgebiete\vg250-ew_12-31.utm32s.gpkg.ebenen\vg250-ew_ebenen_1231\DE_VG250.gpkg"
    BOUNDARY_LAYER = "vg250_lan"
    OUTPUT_DIR = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\DatasetSpecific\CORINE_Bavaria"
    YEARS = [2012, 2015, 2018, 2021]
    SKIP_EXISTING = True

    # ========================================================================
    # OPTION 1: Check directory structure first
    # ========================================================================
    
    check_corine_structure(CORINE_BASE_DIR)
    
    # ========================================================================
    # OPTION 2: Process all years
    # ========================================================================
    
    user_input = input("\nProceed with processing? (y/n): ")
    
    if user_input.lower() == 'y':
        start_time = datetime.now()
        print(f"\nStart: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
        
        # Processing
        results = process_all_corine_years(
            CORINE_BASE_DIR,
            LBM_BASE_DIR,         
            BOUNDARY_FILE,
            OUTPUT_DIR,
            years=YEARS,
            boundary_layer=BOUNDARY_LAYER,
            skip_existing=SKIP_EXISTING

        )
        
        end_time = datetime.now()
        duration = end_time - start_time
        
        print(f"\nEnd: {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"Duration: {duration}")
        print(f"\n✓ Done!")
    else:
        print("\nProcessing cancelled.")


if __name__ == "__main__":
    main()